In [1]:
%load_ext autoreload
%autoreload 2

#### This scrapt is used for match station, remove nan values, and match unit

In [2]:
from crmprtd import Row
import logging
import os
import pickle
import pandas as pd
import numpy as np
from natsort import natsorted
from natsort import natsort_keygen
from pprint import pprint
from collections import Counter
from collections import defaultdict

import re
from rapidfuzz import fuzz
from crmprtd import infer
from fern_func import *
from sql_func import *

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)


In [3]:
# --- Main workflow ---
# --- read data ---
meta_path = '/fern_data/FERNNorth2024_VF/20241125 MeteorologicalNetworks-FERN-VF-shared.xlsx'
data_path = '/fern_data/FERNNorth2024_VF/WxData24'

df_station = pd.read_excel(meta_path)
df_station = df_station.sort_values(by='station_name', key=natsort_keygen())

csv_files = [f for f in os.listdir(data_path) if f.endswith('.csv')]
# Sort in natural order
csv_files = natsorted(csv_files)

# --- output folder ---
output_folder = '/workspaces/crmprtd/fern/all_stations_insert/rows_output/'
os.makedirs(output_folder, exist_ok=True)


In [4]:
df_station['station_name']

0           Atlin School
1               BarrenWx
2            BlackhawkWx
3              BoulderWx
4              BowronPit
5              BulkleyWx
11                 CPFWx
6     Canoe Mountain Stn
7           Caribou Pass
8              ChapmanWx
9            ChiefLakeWx
10            CoalmineWx
12             CrookedLk
13             CrystalWx
14                Dennis
15             DunsterWx
16              EndakoWx
17              GeorgeWx
18              GunnelWx
19           HourglassWx
20       Hudson Bay Mtn2
21                IBB2Wx
22                IBB3Wx
31                Inklin
23               Kluskus
24              MacJxnWx
25          MiddleforkWx
26               NondaWx
27                PinkWx
28              SaxtonWx
29             SeebachWx
30               Sunbeam
32               Tamarac
33            ThompsonWx
34       Willow-BowronWx
Name: station_name, dtype: object

	1.	Try the original name.
	2.	If not found, try removing Wx.
	3.	If still not found, look in the manual corrections dictionary.
	4.	If still no data, record it in stations_no_data.
	5.	Log clearly each attempt and which correction was used.

In [18]:
import sqlalchemy as sa
import pandas as pd
import logging

# --- Setup database connection ---
engine = sa.create_engine("postgresql://tongli1997@pg01.pcic.uvic.ca:5432/crmp")

# --- Setup logger ---
log_path = '/workspaces/crmprtd/fern/all_stations_insert/'
log_file = "fern_db_summary.log"

logger = logging.getLogger("station_logger")
logger.setLevel(logging.INFO)

if logger.hasHandlers():
    logger.handlers.clear()

fh = logging.FileHandler(log_path + log_file, mode="w", encoding="utf-8")
fh.setLevel(logging.INFO)
formatter = logging.Formatter("%(message)s")
fh.setFormatter(formatter)
logger.addHandler(fh)

# --- Column widths for formatting ---
col_widths = {
    "station_name": 25,
    "net_var_name": 25,
    "unit": 12,
    "earliest_time": 22,
    "latest_time": 22,
    "num_records": 10
}

# --- Manual corrections dictionary ---
# Key = original station, Value = (corrected_name, exact_match)
manual_corrections = {
    'Atlin School': ('Atlin', False),
    'BowronPit': ('Bowron Pit', False),
    'ChiefLakeWx': ('CHIEF LAKE', True),
    'MacJxnWx': ('Mackenzie Jxn', False),
    'BoulderWx': ('BOULDER CREEK', True),
    'Dennis': ('Dennis', True),
    'DunsterWx': ('DUNSTER', True)
}

# --- List to store stations still with no data ---
stations_no_data = []

# --- Function to query and log a station ---
def query_and_log(station_pattern, display_name=None, exact_match=False):
    """
    Query a station and log its data.
    
    :param station_pattern: Pattern to search in the DB
    :param display_name: Name to display in the log (optional)
    :param exact_match: If True, use exact match (=), else use ILIKE
    :return: True if data found, None if empty
    """
    display_name = display_name or station_pattern

    where_clause = "m.station_name = :station_pattern" if exact_match else "m.station_name ILIKE :station_pattern"
    pattern_param = station_pattern if exact_match else f"%{station_pattern}%"

    query_text = f"""
        SELECT
            m.station_name,
            v.net_var_name,
            v.unit,
            MIN(o.obs_time) AS earliest_time,
            MAX(o.obs_time) AS latest_time,
            COUNT(*) AS num_records
        FROM obs_raw AS o
        JOIN meta_history AS m
            ON o.history_id = m.history_id
        JOIN meta_vars AS v
            ON o.vars_id = v.vars_id
        WHERE {where_clause}
        GROUP BY m.station_name, v.net_var_name, v.unit
        ORDER BY v.net_var_name;
    """

    query = sa.text(query_text)

    with engine.connect() as conn:
        df = pd.read_sql(query, conn, params={"station_pattern": pattern_param})

    if df.empty:
        return None

    # --- Log table ---
    logger.info(f"\n{'='*140}")
    logger.info(f"📍 Station: {display_name}")
    logger.info(f"{'='*140}")
    logger.info(f"{'Station':{col_widths['station_name']}} "
                f"{'Variable':{col_widths['net_var_name']}} "
                f"{'Unit':{col_widths['unit']}} "
                f"{'Earliest Time':{col_widths['earliest_time']}} "
                f"{'Latest Time':{col_widths['latest_time']}} "
                f"{'Count':{col_widths['num_records']}}")
    logger.info("-" * sum(col_widths.values()))

    for _, row in df.iterrows():
        logger.info(f"{row['station_name']:{col_widths['station_name']}} "
                    f"{row['net_var_name']:{col_widths['net_var_name']}} "
                    f"{(row['unit'] or 'N/A'):{col_widths['unit']}} "
                    f"{str(row['earliest_time']):{col_widths['earliest_time']}} "
                    f"{str(row['latest_time']):{col_widths['latest_time']}} "
                    f"{row['num_records']:{col_widths['num_records']}}")

    # --- Totals ---
    total_var_types = len(df)
    total_var_counts = df['num_records'].sum()
    logger.info("-" * sum(col_widths.values()))
    logger.info(f"{'TOTAL':{col_widths['station_name']}} "
                f"{'':{col_widths['net_var_name']}} "
                f"{'':{col_widths['unit']}} "
                f"{'':{col_widths['earliest_time']}} "
                f"{'':{col_widths['latest_time']}} "
                f"{total_var_counts:{col_widths['num_records']}}")
    logger.info(f"Number of variable types: {total_var_types}\n")

    return True

# --- Main loop for all stations ---
for station_name in df_station['station_name']:
    # 1. First attempt
    if query_and_log(station_name):
        continue

    # 2. Retry removing 'Wx' suffix
    if station_name.endswith('Wx'):
        revised_name = station_name[:-2].strip()
        logger.info(f"🔄 No data for '{station_name}', retrying as '{revised_name}' (LIKE search)...")
        if query_and_log(revised_name, display_name=f"{station_name} (retry: {revised_name})"):
            continue

    # 3. Manual corrections
    if station_name in manual_corrections:
        corrected_name, exact_match = manual_corrections[station_name]
        match_type = "exact match" if exact_match else "LIKE search"
        logger.info(f"🔄 No data for '{station_name}', trying manual correction: '{corrected_name}' ({match_type})...")
        if query_and_log(corrected_name, display_name=f"{station_name} (manual correction: {corrected_name})", exact_match=exact_match):
            continue

    # 4. Still no data
    logger.info(f"❌ No data found for '{station_name}' after all attempts.\n")
    stations_no_data.append(station_name)

logger.info("✅ All stations summarized successfully.")
logger.info(f"Stations with no data after all attempts: {stations_no_data}")

2025-10-30 21:47:58 - INFO - 🔄 No data for 'Atlin School', trying manual correction: 'Atlin' (LIKE search)...
2025-10-30 21:48:00 - INFO - 
2025-10-30 21:48:00 - INFO - 📍 Station: Atlin School (manual correction: Atlin)
2025-10-30 21:48:00 - INFO - ============================================================================================================================================
2025-10-30 21:48:00 - INFO - Station                   Variable                  Unit         Earliest Time          Latest Time            Count     
2025-10-30 21:48:00 - INFO - --------------------------------------------------------------------------------------------------------------------
2025-10-30 21:48:00 - INFO - ATLIN                     air_temp                  celsius      2020-01-23 19:00:00    2025-10-23 19:00:00         40513
2025-10-30 21:48:00 - INFO - ATLIN                     avg_wnd_dir_10m_pst10mts  degrees      2020-01-23 19:00:00    2025-10-23 19:00:00         40513
2025-10-30 